# 04 — Learn the response and profiling as likelihood optimization

We now have a simulator, a learned unbinned density model, and a deliberately imperfect signal model. This notebook solves two different optimization problems with neural networks:

1. **Population response:** physical signal strength $\mu\mapsto(g(\mu),a(\mu))$, the parameters that best fit the simulator population with the imperfect model.
2. **Experiment-level profiling:** the full observed experiment and a tested $\nu\mapsto\widehat\alpha_\nu$, together with a global estimator $(\widehat\nu,\widehat\alpha)$.

Both networks learn by **minimizing the negative log likelihood at their own outputs**. We do not generate numerical-fit labels for training. Direct fits are reserved for validation. The second task uses ordinary full unbinned experiments; there is no $N=2$ training stage and no quadratic assumption about the likelihood or statistic.

The precedent [Learning Optimal Test Statistics in the Presence of Nuisance Parameters](https://arxiv.org/abs/2203.13079) learns statistics through a classification objective with best-average-power motivation. Our objective here is different: it directly amortizes the specified likelihood optimizations.

## Setup and shared run directory

Run this cell first. Use the same `RUN_NAME` in all five notebooks. Colab can keep the run on Drive; locally, open Jupyter in this directory. Use a fresh kernel when switching from the older exercises. Training and samples remain in the run directory.


In [ ]:
# Run this cell first in each notebook. All five notebooks share RUN_NAME.
import os
import sys
import subprocess
from pathlib import Path

TAG = "sb10_exppoly_10m_slowlr_v3_20261001"  # New tag: rebuild all five stages.
RUN_NAME = TAG
USE_DRIVE = True
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/nsbi_calibration_toolkit")
    BRANCH = "ml4hep_school_tutorial"
    if not (REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH,
            "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git", str(REPO),
        ], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
        subprocess.run([
            "git", "-C", str(REPO), "sparse-checkout", "set", "src",
            "workshops/ml4hep_tifr_colab/calibration",
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    SOURCE = REPO / "workshops/ml4hep_tifr_colab/calibration"
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(SOURCE / "requirements.txt"),
    ], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_BASE = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab/calibration_runs")
    else:
        RUN_BASE = Path("/content/calibration_runs")
else:
    # Open the notebook with calibration/ as its working directory.
    SOURCE = Path.cwd().resolve()
    REPO = SOURCE.parents[2]
    RUN_BASE = SOURCE / "runs"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(SOURCE))
os.chdir(SOURCE)
RUN = Path(os.environ.get("CALIBRATION_RUN", RUN_BASE / RUN_NAME)).resolve()
RUN.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE)
print("TAG:", TAG)
print("Run:", RUN)
if "CALIBRATION_RUN" in os.environ:
    print("CALIBRATION_RUN overrides the tagged path; use a fresh directory for this model.")
print("Use a fresh kernel when switching between the old exercises and calibration.")


In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from model import load_config, with_model_epsilon, fit, nll
from sampling import sample_process
from utils import HybridModel
from inference import (ResponseNetwork, ProfileNetwork, Inference, train_response,
                       train_profiles, make_training_experiments, response_validation,
                       validate_toys, save_inference)

misspecification = json.loads((RUN / "misspecification.json").read_text())
epsilon = float(misspecification["epsilon"])
config = with_model_epsilon(load_config(RUN), epsilon)
assert "morph_normalization" in config, "Run notebook 02 to normalize the exp-poly model first."
device = "cuda" if torch.cuda.is_available() else "cpu"
hybrid = HybridModel.load(RUN, device=device)
OUT = RUN / "inference"
OUT.mkdir(parents=True, exist_ok=True)
spec = {"config": config, "epsilon": epsilon, "hybrid_model_id": hybrid.model_id}
spec_path = OUT / "run_spec.json"
if spec_path.exists() and json.loads(spec_path.read_text()) != spec:
    raise ValueError("The model changed. Use a new RUN directory for this inference study.")
spec_path.write_text(json.dumps(spec, indent=2))
print("Device:", device, "| epsilon:", epsilon, "| density model:", hybrid.model_id)
print("Nominal yields S, B:", config["signal_yield"], config["background_yield"])

## 1. The population response network

Let $R_c(x;\alpha)$ denote a complete process/reference ratio in the **bad likelihood**, and write its intensity ratio as

$$I(x;\nu,\alpha)=\nu S R_S(x;\alpha)+B R_B(x;\alpha).$$

Our target is

$$(g(\mu),a(\mu))=\arg\min_{\nu,\alpha}\left[\nu S+B-\mu S\,\mathbb E_{p_{S,\mathrm{sim}}}\log I(X;\nu,\alpha)-B\,\mathbb E_{p_{B,\mathrm{sim}}}\log I(X;\nu,\alpha)+\frac{\alpha^2}{2}\right].$$

The generating nuisance is fixed at $\alpha_{\mathrm{gen}}=0$. A Gaussian constraint enters the likelihood, while the specified toy law is $Y\sim\mathrm{Uniform}[-2,2]$. Because $\mathbb E[Y]=0$, averaging $(Y-\alpha)^2/2$ leaves $\alpha^2/2$ plus a parameter-independent constant. This is the response for that explicitly chosen ensemble, not the usual Gaussian-auxiliary ensemble.

The response NN receives **only $\mu$** and outputs $(\nu,\alpha)$. The density ratios enter its loss. A vector of physical parameters would simply become a vector input; varying generating nuisances would require including them in the input too.

Notebook 03 saved separate physical signal and background samples, with the six good anchor ratios evaluated on them. Their weighted averages approximate the two simulator expectations above. The likelihood first evaluates each normalized exp-poly process density, then mixes signal and background with the configured misspecification. These are not samples from the learned good density model. We use the whole finite bank at each step, sample only $\mu$, and subtract a parameter-independent baseline from the loss for easier plotting. Bounds are $0\leq\nu\leq3$ and $-1\leq\alpha\leq1$; the sigmoid/tanh outputs approach their endpoints, so endpoint accuracy is checked below.

In [ ]:
bank_metadata = json.loads((RUN / "misspecified/response_bank.json").read_text())
assert bank_metadata["epsilon"] == epsilon
assert bank_metadata["hybrid_model_id"] == hybrid.model_id
signal_bank = np.load(RUN / "misspecified/response_signal.npy")
background_bank = np.load(RUN / "misspecified/response_background.npy")
print("Population quadrature events:", len(signal_bank), "signal +", len(background_bank), "background")

RETRAIN_RESPONSE = False
RESPONSE_STEPS = 2000
response_path = OUT / f"response_{RESPONSE_STEPS}steps.pt"
response_history_path = response_path.with_suffix(".json")
if RETRAIN_RESPONSE or not response_path.exists():
    response_network, response_history = train_response(
        signal_bank, background_bank, config, steps=RESPONSE_STEPS,
        batch_mu=8, device=device, seed=config["seed"] + 4101)
    torch.save(response_network.cpu().state_dict(), response_path)
    response_history_path.write_text(json.dumps(response_history, indent=2))
else:
    response_network = ResponseNetwork(config)
    response_network.load_state_dict(torch.load(response_path, map_location="cpu", weights_only=True))
    response_history = json.loads(response_history_path.read_text())
response_network = response_network.to(device)

history = pd.DataFrame(response_history)
plt.plot(history.step, history.monitor)
plt.xlabel("Training step")
plt.ylabel("Population NLL minus fixed baseline")
plt.title("Fixed mu-grid monitor on the training quadrature bank")
plt.show()

### Separate optimization error from finite-population Monte Carlo error

A direct fit on the **same** bank tests whether the NN minimized its training objective. Fits on **three independent physical banks** show how sensitive that result is to the Monte Carlo approximation of the simulator expectation. The demonstration now uses $S/B=1/10$ ($S=100$, $B=1000$), but finite-bank precision must still be checked.

The independent banks and their fits are diagnostics only: they never train the response NN or select its checkpoint. The shaded range below is the minimum to maximum across three banks, **not a confidence interval**. If same-bank NLL gaps are small but independent fits disagree appreciably, increasing network capacity will not remove that integration uncertainty. Larger or additional banks are then the appropriate next check.

In [ ]:
N_RESPONSE_VALIDATION = 200_000  # per process, in each independent bank
N_RESPONSE_BANKS = 3
mu_grid = np.linspace(*config["mu_range"], 7)
response_train_check = pd.DataFrame(response_validation(
    response_network, signal_bank, background_bank, mu_grid, config))
response_train_check.to_csv(OUT / "response_same_bank.csv", index=False)

independent_checks = []
for bank_index in range(N_RESPONSE_BANKS):
    validation_bank_path = OUT / f"response_validation_{N_RESPONSE_VALIDATION}_{bank_index}.npz"
    if not validation_bank_path.exists():
        rng = np.random.default_rng(config["seed"] + 4201 + bank_index)
        validation_signal = hybrid.anchors(sample_process("signal", N_RESPONSE_VALIDATION, rng))
        validation_background = hybrid.anchors(sample_process("background", N_RESPONSE_VALIDATION, rng))
        np.savez(validation_bank_path, signal=validation_signal, background=validation_background)
    else:
        with np.load(validation_bank_path) as bank:
            validation_signal, validation_background = bank["signal"], bank["background"]
    check = pd.DataFrame(response_validation(
        response_network, validation_signal, validation_background, mu_grid, config))
    check["bank"] = bank_index
    independent_checks.append(check)
    print(f"Independent response bank {bank_index + 1}/{N_RESPONSE_BANKS} fitted.")
response_check = pd.concat(independent_checks, ignore_index=True)
response_check.to_csv(OUT / "response_independent_banks.csv", index=False)
response_spread = response_check.groupby("mu")[["nu_exact", "alpha_exact", "nll_gap"]].agg(["mean", "std", "min", "max"])
display(response_spread)
print("Direct-fit success fraction on independent banks:", response_check.fit_success.mean())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name, label in [(axes[0], "nu", r"$g(\mu)$"), (axes[1], "alpha", r"$a(\mu)$")]:
    spread = response_spread[name + "_exact"]
    ax.plot(mu_grid, response_train_check[name + "_nn"], label="Response NN")
    ax.plot(mu_grid, response_train_check[name + "_exact"], "o", label="Direct: training bank")
    ax.plot(mu_grid, spread["mean"], "x--", label="Direct: independent-bank mean")
    ax.fill_between(mu_grid, spread["min"], spread["max"], alpha=.2,
                    label=f"Range of {N_RESPONSE_BANKS} independent banks")
    ax.set(xlabel=r"$\mu$", ylabel=label)
axes[0].legend(fontsize=8)
axes[1].axhline(0, color="gray", lw=.7)
axes[2].plot(mu_grid, response_train_check.nll_gap, "o-", label="Training bank")
for bank_index, check in response_check.groupby("bank"):
    axes[2].plot(check.mu, check.nll_gap, "x--", alpha=.7, label=f"Independent {bank_index + 1}")
axes[2].axhline(0, color="gray", lw=.7)
axes[2].set(xlabel=r"$\mu$", ylabel="NN NLL − direct minimum")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Represent the full experiment and learn both optimizers

Each event contributes the six normalized good-model anchor ratios

$$\mathbf R(x)=(R_S^-,R_S^0,R_S^+,R_B^-,R_B^0,R_B^+).$$

Together with the known rates, count $N$, and auxiliary $y$, these determine the likelihood under the normalized exp-poly interpolation, with its fixed normalization coefficients and the configured signal–background mixture. The event encoder $\phi$ receives $\log(1+\mathbf R(x))$ and learns a permutation-invariant experiment representation:

$$z(D)=\left[\frac{\sum_{i=1}^N\phi(\mathbf R(x_i))-N\,\overline\phi_{\mathrm{center}}}{\sqrt{\max(N,1)}},\frac{N-B}{\sqrt B},y\right].$$

This is a centered, scaled mean pool with $N$ retained. Centering uses a fixed subset of the background bank and is recomputed through the trainable encoder; it changes conditioning, not which events are included. The empty set has a zero event summary.

The global head receives $z$ and outputs $(\widehat\nu,\widehat\alpha)$. The conditional head also receives a tested **$\nu$** and outputs $\widehat\alpha_\nu$. The fitted nuisance is an output, not an externally supplied input. With several POIs/nuisances, these inputs and outputs become vectors.

We train the encoder and both heads together. Write $G$ for the NLL at the global head, $C_q$ for the conditional NLL at a training query, and $C_{\rm own}$ for the conditional-head NLL at the global head's own POI. The objective is

$$\mathcal L=\mathbb E_D\left[G+C_q+\tfrac12 C_{\rm own}+\mathcal R\right],$$

$$\mathcal R=[C_{\rm own}-\operatorname{sg}(G)]_+^2+[G-\operatorname{sg}(C_{\rm own})]_+^2,$$

where $[u]_+=\max(u,0)$ and $\operatorname{sg}$ stops the gradient. The own-POI query is detached as well. This gives extra conditional training near the learned minimum. The consistency term pushes the **worse** of the two candidate fits down in NLL; its gradient does not pull the better candidate up to match it. This squared penalty concerns agreement between candidate NLLs, not a quadratic assumption about the likelihood as a function of parameters.

Every NLL sums **all events in each training experiment**, including the count/rate term and Gaussian constraint once per likelihood evaluation. A parameter-independent baseline is subtracted for numerical conditioning. No scalar test-statistic labels or optimized-fit labels enter training. A finite learned pool can still discard useful information, so direct-fit validation is essential.

### Training ensembles and cost

We use independent experiments from both the physical simulator and the bad reference model so the optimizer encounters both domains. This helps cover the two distributions used in notebook 05. Generating nuisances span $[-1,1]$ during training; the calibration study remains at physical $\alpha_{\mathrm{gen}}=0$.

The physical simulator now varies its scale continuously with $\alpha$. The learned model uses normalized exp-poly interpolation between its three anchors. They agree on the meaning of the anchors, but an intermediate-nuisance interpolation remains a modeling approximation. Reference training toys resample the integration bank using the **same normalized exp-poly density as the fitted model**, including the signal misspecification. This is an explicit finite-bank approximation. Final validation below uses **new physical unbinned events**.

The default is **6,000 training experiments per source**, 256 monitor experiments per source, and 120 epochs. At $S/B=1/10$, $N$ is around 1,100 rather than 10,100, so the larger experiment count still uses fewer total events than the previous setup. The six float32 anchors occupy roughly 0.3 GB for the default training cache, before Python/Torch overhead. Training-toy caches and checkpoints are keyed by sample count.

Training uses learning-rate decay and retains the best held-out checkpoint. The fixed training and held-out monitors use the same nine-point POI grid, including both boundaries. Stochastic training queries also concentrate near the generating POI and the network's current estimate. Their absolute NLL levels can still differ because the experiments and parameter-independent baselines differ; the curves are optimization diagnostics, not direct estimates of regret. The fresh numerical fits below measure global and conditional regret separately.

In [ ]:
centering_anchors = background_bank[:2048]
N_TRAIN_PER_SOURCE = 6000
N_MONITOR_PER_SOURCE = 256
PROFILE_EPOCHS = 120
RETRAIN_PROFILES = False

toys_path = OUT / f"training_experiments_{N_TRAIN_PER_SOURCE}_{N_MONITOR_PER_SOURCE}.pt"
if not toys_path.exists():
    training_toys = make_training_experiments(
        hybrid, config, epsilon, N_TRAIN_PER_SOURCE,
        np.random.default_rng(config["seed"] + 4301))
    monitor_toys = make_training_experiments(
        hybrid, config, epsilon, N_MONITOR_PER_SOURCE,
        np.random.default_rng(config["seed"] + 4302))
    torch.save({"training": training_toys, "monitor": monitor_toys}, toys_path)
else:
    cached_toys = torch.load(toys_path, map_location="cpu", weights_only=False)
    training_toys, monitor_toys = cached_toys["training"], cached_toys["monitor"]
print("Training experiments:", len(training_toys), "| monitoring experiments:", len(monitor_toys))
print("Cached events:", sum(len(t["anchors"]) for t in training_toys))

profile_path = OUT / f"profiles_{N_TRAIN_PER_SOURCE}toys_{PROFILE_EPOCHS}epochs.pt"
profile_history_path = profile_path.with_suffix(".json")
if RETRAIN_PROFILES or not profile_path.exists():
    profile_network, profile_history = train_profiles(
        training_toys, monitor_toys, centering_anchors, config,
        epochs=PROFILE_EPOCHS, batch_size=16, device=device,
        seed=config["seed"] + 4303)
    torch.save(profile_network.cpu().state_dict(), profile_path)
    profile_history_path.write_text(json.dumps(profile_history, indent=2))
else:
    profile_network = ProfileNetwork(centering_anchors, config)
    profile_network.load_state_dict(torch.load(profile_path, map_location="cpu", weights_only=True))
    profile_history = json.loads(profile_history_path.read_text())

inference = Inference(response_network, profile_network, config, epsilon, device)
save_inference(RUN, inference, metadata={**spec,
    "training_experiments": len(training_toys),
    "monitor_experiments": len(monitor_toys),
    "response_population_per_process": len(signal_bank),
    "auxiliary_law": "Uniform[-2,2]", "physical_calibration_alpha": 0.0,
    "reference_training": "finite integration bank",
    "statistic": "signed full-likelihood gap at amortized optima"})

hist = pd.DataFrame(profile_history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(hist.epoch, hist.train_step, alpha=.5, label="Stochastic training batches")
axes[0].plot(hist.epoch, hist.train, label="Fixed training monitor")
axes[0].plot(hist.epoch, hist.validation, label="Fixed held-out monitor")
axes[0].set(xlabel="Epoch", ylabel="Centered objective", title="Matched fixed monitors; stochastic training queries")
axes[0].legend(fontsize=8)
for column, label in [("validation_global", "Global"), ("validation_conditional", "Conditional query"),
                      ("validation_own_conditional", "Conditional at own POI")]:
    if column in hist:
        axes[1].plot(hist.epoch, hist[column] - hist[column].iloc[0], label=label)
axes[1].set(xlabel="Epoch", ylabel="Change from first epoch", title="Held-out objective components")
if axes[1].lines:
    axes[1].legend()
if "learning_rate" in hist:
    axes[2].semilogy(hist.epoch, hist.learning_rate)
axes[2].set(xlabel="Epoch", ylabel="Learning rate", title="Decay and best-checkpoint selection")
plt.tight_layout()
plt.show()

## 3. Obtain a statistic without an iterative profile fit

For any tested $\nu$, evaluate the original likelihood at the two predicted optima:

$$t_{\mathrm{raw}}(\nu;D)=2\left[\mathrm{NLL}(\nu,\widehat\alpha_\nu(D);D)-\mathrm{NLL}(\widehat\nu(D),\widehat\alpha(D);D)\right].$$

The exact constrained fits would make this nonnegative. Approximate optimizers can produce negative values. We retain and report these **signed optimization gaps** here; no clipping conceals an optimization failure.

At evaluation, a fixed candidate list includes $\alpha=-1,0,+1$, network predictions, and a 13-point POI grid including both boundaries. The global candidate set is independent of the requested scan points, so asking for a denser scan cannot change the reported global estimate. The selected global nuisance is also a conditional candidate at every tested POI, giving a continuous extra candidate curve and enforcing consistency at the selected global POI. We report the raw head inconsistency separately. These checks allow exact endpoints and the nominal nuisance value to win despite bounded smooth NN outputs; they are not an iterative minimization and do not guarantee the global optimum.

The encoder and the final likelihood evaluation still scale with the number of events. A scan reuses one experiment embedding, but each query evaluates the full likelihood. What we amortize is the search for fitted parameters. We do not claim constant-cost likelihood evaluation or an exact sufficient learned embedding.

## 4. Validate on actual unbinned experiments

The checks below compare to direct bounded fits of the same likelihood. We inspect global and conditional NLL excess **separately**, because errors in the two optima can cancel in their difference. We also inspect fitted parameters, statistic tails, negative gaps, and full nonlinear scan shapes. The finite-candidate rule makes the statistic vanish at its selected global estimate; we also report the conditional head's value there **before** adding that consistency candidate, to expose errors that this safeguard could hide.

The timing comparison excludes simulation and ratio evaluation for both methods, and includes embedding and final likelihood evaluation for the NN. This toy has only two fitted parameters, so a numerical fit may already be fast; any speed advantage must be measured.

In [ ]:
VALIDATION_MU = [0., 0.25, 1., 2.]
N_VALIDATION_PER_MU = 64  # 256 independent full experiments
validation_rows, scan_examples = validate_toys(
    inference, hybrid, config, epsilon, VALIDATION_MU, N_VALIDATION_PER_MU,
    np.random.default_rng(config["seed"] + 4401), alpha_gen=0.)
validation = pd.DataFrame(validation_rows)
validation["statistic_error"] = validation.statistic_nn - validation.statistic_exact
validation["global_alpha_error"] = validation.alpha_nn - validation.alpha_exact
validation["conditional_alpha_error"] = validation.conditional_alpha_nn - validation.conditional_alpha_exact
validation.to_csv(OUT / "unbinned_validation.csv", index=False)
torch.save(scan_examples, OUT / "scan_examples.pt")

summary = pd.DataFrame([{
    "mu": mu, "toys": len(group),
    "median_abs_T_error": group.statistic_error.abs().median(),
    "p95_abs_T_error": group.statistic_error.abs().quantile(.95),
    "median_global_NLL_gap": group.global_gap.median(),
    "p95_global_NLL_gap": group.global_gap.quantile(.95),
    "median_conditional_NLL_gap": group.conditional_gap.median(),
    "p95_conditional_NLL_gap": group.conditional_gap.quantile(.95),
    "negative_T_fraction": (group.statistic_nn < 0).mean(),
    "minimum_raw_T": group.statistic_nn.min(),
    "p95_abs_T_at_own_minimum": group.statistic_at_own_minimum.abs().quantile(.95),
    "p95_abs_raw_head_T_at_global": group.conditional_head_T_at_global.abs().quantile(.95),
    "p95_abs_candidate_T_at_global": group.candidate_T_at_global.abs().quantile(.95),
    "direct_fit_success_fraction": group.fit_success.mean(),
} for mu, group in validation.groupby("mu")])
summary.to_csv(OUT / "unbinned_validation_summary.csv", index=False)
display(summary)
print("Median NN evaluation seconds:", validation.nn_seconds.median())
print("Median direct-fit seconds:", validation.exact_seconds.median())
print("Median time ratio direct / NN:", (validation.exact_seconds / validation.nn_seconds).median())
print("Negative raw gaps:", int((validation.statistic_nn < 0).sum()), "/", len(validation))

# A failed direct fit, or a negative regret beyond roundoff, makes the numerical
# benchmark suspect. Keep those toys visible rather than dropping them silently.
suspect = (~validation.fit_success.astype(bool)
           | (validation.global_gap < -1e-5) | (validation.conditional_gap < -1e-5))
print("Toys requiring a closer direct-fit check:", int(suspect.sum()))
if suspect.any():
    display(validation.loc[suspect, ["mu", "global_gap", "conditional_gap", "fit_success"]].head(20))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
points = axes[0, 0].scatter(validation.statistic_exact, validation.statistic_nn, c=validation.mu, s=15)
limit = max(validation.statistic_exact.max(), validation.statistic_nn.max(), 1.)
axes[0, 0].plot([0, limit], [0, limit], "k--")
axes[0, 0].set(xlabel="Direct unbinned t", ylabel="NN signed t")
axes[0, 1].scatter(validation.statistic_exact, validation.statistic_error, c=validation.mu, s=15)
axes[0, 1].axhline(0, color="k", ls="--")
axes[0, 1].set(xlabel="Direct unbinned t", ylabel="NN t − direct t")
axes[0, 2].scatter(validation.global_gap, validation.conditional_gap, c=validation.mu, s=15)
regret_range = [min(validation.global_gap.min(), validation.conditional_gap.min(), 0.),
                max(validation.global_gap.max(), validation.conditional_gap.max(), .01)]
axes[0, 2].plot(regret_range, regret_range, "k--", label="Equal regrets: t errors cancel")
axes[0, 2].set(xlabel="Global NLL regret", ylabel="Conditional NLL regret")
axes[0, 2].legend(fontsize=8)
axes[1, 0].scatter(validation.nu_exact, validation.nu_nn, c=validation.mu, s=15)
axes[1, 0].plot(config["nu_bounds"], config["nu_bounds"], "k--")
axes[1, 0].set(xlabel="Direct fitted nu", ylabel="NN fitted nu")
for ax, name, residual in [(axes[1, 1], "alpha", "global_alpha_error"),
                            (axes[1, 2], "conditional_alpha", "conditional_alpha_error")]:
    ax.scatter(validation[name + "_exact"], validation[residual], c=validation.mu, s=15)
    ax.axhline(0, color="k", ls="--")
    # Data-driven axes expose the small nuisance residuals near zero.
    ax.set(xlabel="Direct " + name, ylabel="NN − direct " + name)
plt.tight_layout()
plt.show()

### Boundary and tail summaries

Boundary fits and large statistics deserve separate attention. The numerical boundary tolerance below only labels results for this diagnostic; it does not alter any fit or statistic. Small samples in the top tail give noisy summaries, so increase the independent validation sample before making a quantitative tail-accuracy claim.

In [ ]:
boundary = (np.isclose(validation.nu_exact, config["nu_bounds"][0], atol=1e-5)
            | np.isclose(validation.nu_exact, config["nu_bounds"][1], atol=1e-5)
            | np.isclose(np.abs(validation.alpha_exact), 1., atol=1e-5)
            | np.isclose(np.abs(validation.conditional_alpha_exact), 1., atol=1e-5))
tail_threshold = validation.statistic_exact.quantile(.90)
subsets = {"all": np.ones(len(validation), dtype=bool),
           "boundary fits": boundary,
           "top 10% direct t": validation.statistic_exact >= tail_threshold}
display(pd.DataFrame([{"subset": name, "toys": int(mask.sum()),
    "mean_abs_T_error": validation.loc[mask, "statistic_error"].abs().mean(),
    "max_abs_T_error": validation.loc[mask, "statistic_error"].abs().max(),
    "mean_global_NLL_gap": validation.loc[mask, "global_gap"].mean(),
    "mean_conditional_NLL_gap": validation.loc[mask, "conditional_gap"].mean()}
    for name, mask in subsets.items()]))

### Regret, statistic error, and consistency as functions of the physical POI

The relation $t_{\rm NN}-t_{\rm direct}=2(\Delta\mathrm{NLL}_{\rm cond}-\Delta\mathrm{NLL}_{\rm global})$ explains why a good-looking statistic alone can hide inaccurate optimizers. We therefore show each regret separately. Tail errors are ordered by the direct statistic; the top decile here is a diagnostic subset, not a precision measurement of a small tail probability.

The own-minimum check is displayed on its own scale. A finite candidate consistency safeguard may make it zero by construction; that does not certify either head as a numerical optimum. Numerical-fit status remains visible in the summary above.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for name, label, marker in [("global_gap", "Global", "o"),
                             ("conditional_gap", "Conditional", "s")]:
    grouped = validation.groupby("mu")[name]
    axes[0, 0].plot(summary.mu, grouped.median(), marker + "-", label=label + " median")
    axes[0, 0].plot(summary.mu, grouped.quantile(.95), marker + "--", label=label + " 95th percentile")
axes[0, 0].axhline(0, color="gray", lw=.7)
axes[0, 0].set(xlabel=r"$\mu$", ylabel="NLL regret")
axes[0, 0].legend(fontsize=8)
axes[0, 1].plot(summary.mu, summary.median_abs_T_error, "o-", label="Median")
axes[0, 1].plot(summary.mu, summary.p95_abs_T_error, "s--", label="95th percentile")
axes[0, 1].set(xlabel=r"$\mu$", ylabel="Absolute t error")
axes[0, 1].legend()
axes[0, 2].plot(summary.mu, summary.negative_T_fraction, "o-")
axes[0, 2].set(xlabel=r"$\mu$", ylabel="Fraction with signed t < 0",
               ylim=(-.005, max(.05, min(1., 1.2 * summary.negative_T_fraction.max()))))
axes[1, 0].scatter(validation.statistic_exact, validation.statistic_error.abs(), c=validation.mu, s=15)
axes[1, 0].axvline(tail_threshold, color="k", ls="--", label="Direct t 90th percentile")
axes[1, 0].set(xlabel="Direct t (tail diagnostic)", ylabel="Absolute t error")
axes[1, 0].legend(fontsize=8)
axes[1, 1].scatter(validation.mu, validation.conditional_head_T_at_global, s=15, alpha=.5, label="Conditional head")
axes[1, 1].scatter(validation.mu, validation.statistic_at_own_minimum, s=15, marker="x", label="After candidate consistency")
axes[1, 1].axhline(0, color="k", ls="--")
axes[1, 1].set(xlabel=r"$\mu$", ylabel="Signed t at NN global estimate")
axes[1, 1].legend(fontsize=8)
for name, label in [("global_alpha_error", "Global"), ("conditional_alpha_error", "Conditional")]:
    grouped = validation.assign(abs_error=validation[name].abs()).groupby("mu").abs_error
    axes[1, 2].plot(summary.mu, grouped.quantile(.95), "o-", label=label)
axes[1, 2].set(xlabel=r"$\mu$", ylabel=r"95th percentile $|\widehat\alpha_{\rm NN}-\widehat\alpha_{\rm direct}|$")
axes[1, 2].legend()
plt.tight_layout()
plt.show()

# Report numerical-benchmark status separately; retain every toy in the figures.
display(validation.groupby("fit_success").agg(
    toys=("mu", "size"), mean_global_regret=("global_gap", "mean"),
    mean_conditional_regret=("conditional_gap", "mean"),
    median_abs_T_error=("statistic_error", lambda s: s.abs().median())))

In [ ]:
nu_scan = np.linspace(*config["nu_bounds"], 61)
fig, axes = plt.subplots(1, len(scan_examples), figsize=(5 * len(scan_examples), 4))
scan_records = []
for ax, toy in zip(np.atleast_1d(axes), scan_examples):
    estimate = inference.evaluate(toy["anchors"], toy["auxiliary"], nu_scan)
    best = fit(toy["anchors"], toy["auxiliary"], config)
    exact = np.asarray([2 * (fit(toy["anchors"], toy["auxiliary"], config, fixed_nu=nu)["nll"] - best["nll"])
                        for nu in nu_scan])
    ax.plot(nu_scan, exact, label="Direct full likelihood")
    ax.plot(nu_scan, estimate["statistic"], "--", label="Amortized optima")
    ax.axhline(0, color="gray", lw=.7)
    ax.set(xlabel=r"$\nu$", ylabel=r"$t(\nu)$", title=f'Physical mu={toy["mu"]:g}, y={toy["auxiliary"]:.2f}')
    scan_records.append({"mu": toy["mu"], "nu": nu_scan.tolist(), "direct_t": exact.tolist(),
                         "nn_raw_t": estimate["statistic"].tolist()})
axes[0].legend()
plt.tight_layout()
plt.show()
(OUT / "scan_validation.json").write_text(json.dumps(scan_records, indent=2))

### Inspect the nuisance likelihood through the nominal point

For the first independent experiment at each physical $\mu$, hold $\nu$ at the tested response value and scan $\alpha$ around zero and the fitted nuisances. This exposes the local width and whether the conditional NN resolves its minimum. The normalized exp-poly likelihood should be smooth through zero. The derivative is a finite-difference plotting diagnostic; fits and training use the likelihood's derivatives, including its normalization.

In [ ]:
fig, axes = plt.subplots(2, len(scan_examples), figsize=(5 * len(scan_examples), 7), squeeze=False)
nuisance_records = []
for column, toy in enumerate(scan_examples):
    query = float(inference.response(toy["mu"])[0])
    direct = fit(toy["anchors"], toy["auxiliary"], config, fixed_nu=query)
    estimate = inference.evaluate(toy["anchors"], toy["auxiliary"], query)
    fitted = [direct["alpha"], float(estimate["conditional_alpha"])]
    window = min(1., max(.025, 1.5 * max(abs(a) for a in fitted)))
    alpha_scan = np.linspace(-window, window, 201)
    values = np.asarray([nll(toy["anchors"], query, a, toy["auxiliary"], config) for a in alpha_scan])
    delta = values - direct["nll"]
    axes[0, column].plot(alpha_scan, delta, label="Full conditional NLL")
    axes[0, column].axvline(direct["alpha"], color="C1", label="Direct minimum")
    axes[0, column].axvline(estimate["conditional_alpha"], color="C2", ls="--", label="NN candidate")
    axes[0, column].set(ylabel="NLL − direct minimum", title=f'Physical mu={toy["mu"]:g}, tested nu={query:.3f}')
    axes[1, column].plot(alpha_scan, np.gradient(delta, alpha_scan, edge_order=2))
    axes[1, column].axhline(0, color="gray", lw=.7)
    axes[1, column].set(xlabel=r"$\alpha$", ylabel=r"$\partial\mathrm{NLL}/\partial\alpha$ (finite difference)")
    for ax in axes[:, column]:
        ax.axvline(0, color="gray", ls=":", lw=.8)
    nuisance_records.append({"mu": toy["mu"], "query_nu": query, "alpha": alpha_scan.tolist(),
                             "delta_nll": delta.tolist(), "alpha_direct": direct["alpha"],
                             "alpha_nn": float(estimate["conditional_alpha"])})
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()
(OUT / "nuisance_scan_validation.json").write_text(json.dumps(nuisance_records, indent=2))

## What to inspect before the distribution flows

- **Response:** same-bank gaps diagnose optimization; the spread across independent population fits measures sensitivity to finite-bank precision. Inspect both $g$ and the fitted nuisance $a$.
- **Profiling:** compare conditional/global NLL excess separately, including nuisance boundaries and the region near zero. If training improves but held-out gaps do not, add training experiments; if both remain poor, change optimization or encoder/head capacity.
- **Statistic:** inspect signed gaps and scan shapes, especially tails. A small average error does not establish inference accuracy at a particular tail probability.
- **Cost:** record the measured speed rather than assuming the NN helps in this simple two-parameter example.

Notebook 05 calibrates the distribution of the learned statistic on separate toys. It explicitly defines $T_{\mathrm{NN}}=\max(0,t_{\mathrm{raw}})$ and models its point mass at zero; optimization-induced zeros are tracked separately from genuine constrained-fit effects. Calibrating that distribution can yield valid tests for the chosen toy ensemble, but it does not turn an inaccurate optimizer into the exact profile likelihood. The raw-gap and direct-fit checks in this notebook remain necessary.